In [1]:
import re
import pandas as pd
from tqdm import tqdm
import unicodedata
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
tqdm.pandas()

In [3]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)

    # Нормализуем Unicode (заменяем нестандартные пробелы и символы)
    text = unicodedata.normalize("NFKC", text)

    # HTML / Markdown мусор
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"[#*_`~>\[\]]+", " ", text)

    # Эмодзи и спецсимволы
    text = re.sub(
        r"[" 
        r"\U0001F600-\U0001F64F"
        r"\U0001F300-\U0001F5FF"
        r"\U0001F680-\U0001F6FF"
        r"\U0001F1E0-\U0001F1FF"
        r"★☆•●▶️⛔️✅✳️✴️✨☑️❌❗️❕❓❔№📌🔹🔸🔺🫶🤙🧸"
        r"]+", " ", text
    )

    # Артефакты PDF, JSON, HTML
    text = re.sub(r"\(cid:\d+\)", " ", text)
    text = re.sub(r"\{.*?\}|\[.*?\]", " ", text)

    # JSON и CSS-вставки
    text = re.sub(r'"[a-zA-Z0-9_-]+"\s*:\s*".*?"', " ", text)

    # Контактные данные
    text = re.sub(r"[\w\.-]+@[\w\.-]+", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\+7\s?\(?\d{3}\)?[\d\s\-]{5,}", " ", text)
    text = re.sub(r"\b\d{5,6}\b", " ", text)

    # Технические / шаблонные слова
    text = re.sub(r"(редакци[ияи]|приложени[ея]|утвержден[ао]|приказ[а-я]*)", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"(Подпись|Ф\.И\.О\.|Дата заполнения|Дата принятия)", " ", text, flags=re.IGNORECASE)

    # Строки-заглушки
    text = re.sub(r"(страница не найдена|ошибка загрузки|загрузите приложение|вернитесь на главную)", " ", text, flags=re.IGNORECASE)

    # Навигационные / меню / служебные теги
    text = re.sub(r"(Название тарифа|Стоимость|Интернет|Звонки|СМС|₽|ГБ|мин)", " ", text)
    text = re.sub(r"\|{2,}", " ", text)
    text = re.sub(
        r"(Общая информация|Мобильное и Альфа Онлайн|Комиссии по карте|Перевыпуск|Начало использования|"
        r"Как пользоваться картой|Всё про кэшбэк|Справки и выписки|Программа|Изготовление и доставка|"
        r"Полезное о продуктах|Поиск по сайту|Найти|NEW PAGE VALUE|default value|geo value|future value|"
        r"Информация об изменениях|smoke|PAGE VALUE|default geo value)",
        " ", text, flags=re.IGNORECASE
    )

    # Если вся строка — техническая / поисковая
    if re.search(r"(Поиск по сайту|Найти|PAGE VALUE|default value|geo value|smoke)", text, flags=re.IGNORECASE):
        return ""

    # Номера, даты, приказы
    text = re.sub(r"\b\d+(\.\d+)*\.?\b", " ", text)
    text = re.sub(r"№\s?\d+", " ", text)

    # Избыточные цифры или валюты
    if len(re.findall(r"[0-9₽%]", text)) / (len(text) + 1e-5) > 0.4:
        return ""

    # Списки, разделители, палки
    text = re.sub(r"(\|+|\–+|\—+|\-+|_+|=+)", " ", text)
    text = re.sub(r"[-–•▪●■]+", " ", text)

    # Удалить китайский / японский / корейский
    text = re.sub(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]+", " ", text)

    # Латиница (если почти весь текст на ней)
    if len(re.findall(r"[A-Za-z]", text)) / (len(text) + 1e-5) > 0.8:
        return ""

    # Информационные блоки
    text = re.sub(r"(Информация об изменениях[:]?.*?)(?=$|\n)", " ", text, flags=re.IGNORECASE)

    # Короткие слова и мусор
    text = re.sub(r"\b\d{1,3}\b", " ", text)
    text = re.sub(r"\b[а-яА-ЯёЁ]{1,2}\b", " ", text)

    # Повторы знаков препинания
    text = re.sub(r"[.]{2,}", ".", text)
    text = re.sub(r"[,]{2,}", ",", text)
    text = re.sub(r"[!?]{2,}", "!", text)

    # Повторы пробелов и переводов строк
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)

    # Очистка “заголовков” и списков
    text = re.sub(r"^([А-ЯЁ][а-яё]+(\s|,|\.|!|\?)){5,}", " ", text)

    # Удалить placeholder символы ▯□�☒░▒...
    text = re.sub(r"[▯□�☒░▒■□▪▫◻️◼️]+", " ", text)

    # Удалить строки из одних пробелов или мусора
    text = text.strip()
    if len(text) < 40 or text.isspace():
        return ""

    return text

In [3]:
websites = pd.read_csv("../data/websites_updated.csv")
websites.head()

,web_id,url,kind,title,text
0,1,https://alfabank.ru/,html,"Альфа-Банк - кредитные и дебетовые карты, кред...",Рассчитайте выгоду\nРасчёт калькулятора предва...
1,2,https://alfabank.ru/a-club/,html,А-Клуб. Деньги имеют значение,Брокерские услуги\nОткрытие брокерского счёта ...
2,3,https://alfabank.ru/a-club/ultimate/,html,А-Клуб. Деньги имеют значение,Хотите получить больше информации?\nПозвоните ...
3,4,https://alfabank.ru/actions/rules/,html,Скидки по картам,Правила проведения Акции «Альфа Пятница. Бараб...
4,5,https://alfabank.ru/alfafuture/,html,Альфа‑Будущее: Платформа для развития студенто...,Образование\nМагистратуры\nМагистратура ВШЭ\nМ...


In [5]:
websites["clean_text"] = websites["text"].progress_apply(clean_text)

100%|██████████| 1938/1938 [00:11<00:00, 167.59it/s]


In [6]:
websites = websites[websites["clean_text"].str.len() > 80]

In [7]:
websites = websites.groupby("web_id", as_index=False)["clean_text"].apply(lambda x: " ".join(set(x)))

In [8]:
websites.to_csv("../data/websites_clean.csv", index=False)

In [27]:
(pd.read_csv("../data/websites_clean.csv")).head()

,web_id,clean_text
0,1,Рассчитайте выгоду Расчёт калькулятора предвар...
1,2,Брокерские услуги Открытие брокерского счёта п...
2,3,Хотите получить больше информации? Позвоните н...
3,4,Правила проведения Акции «Альфа Пятница. Бараб...
4,5,Образование Магистратуры Магистратура ВШЭ Маги...


In [6]:
df = pd.read_csv("../data/websites_clean.csv")
print("Строк:", len(df))
print("Средняя длина:", df["clean_text"].str.len().mean())
print("Минимальная длина:", df["clean_text"].str.len().min())
df.head(3)

Строк: 1849
Средняя длина: 6030.41806381828
Минимальная длина: 85


,web_id,clean_text
0,1,Рассчитайте выгоду Расчёт калькулятора предвар...
1,2,Брокерские услуги Открытие брокерского счёта п...
2,3,Хотите получить больше информации? Позвоните н...


In [8]:
def smart_chunk_text(text, chunk_size=1000, overlap=200, min_len=50):
    try:
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
            separators=["\n\n", ".", "!", "?", ";", ",", " "]
        )
        chunks = splitter.split_text(text)
    except ImportError:
        # fallback на простое деление
        text = str(text).strip()
        if len(text) < min_len:
            return []
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end].strip()
            if len(chunk) >= min_len:
                chunks.append(chunk)
            start += chunk_size - overlap
    return chunks

In [9]:
all_chunks = []

In [10]:
for _, row in df.iterrows():
    text = row["clean_text"]
    web_id = row["web_id"]

    for i, chunk in enumerate(smart_chunk_text(text, chunk_size=1000, overlap=200)):
        all_chunks.append({
            "web_id": web_id,
            "chunk_id": f"{web_id}_{i}",
            "chunk_text": chunk
        })

In [13]:
chunks_df = pd.DataFrame(all_chunks)
chunks_df.drop_duplicates(subset=["chunk_text"], inplace=True)

In [15]:
chunks_df["len"] = chunks_df["chunk_text"].str.len()
print(chunks_df["len"].describe())

count    11435.000000
mean       803.742020
std        222.154291
min          1.000000
25%        729.000000
50%        899.000000
75%        959.000000
max       1000.000000
Name: len, dtype: float64


In [16]:
chunks_df = chunks_df[chunks_df["len"] > 100].reset_index(drop=True)

In [17]:
print(f"✅ Осталось {len(chunks_df)} чанков после фильтрации")
chunks_df["len"].describe()

✅ Осталось 11366 чанков после фильтрации


count    11366.000000
mean       808.195495
std        215.317328
min        101.000000
25%        735.000000
50%        900.000000
75%        959.000000
max       1000.000000
Name: len, dtype: float64

In [18]:
chunks_df.to_csv("../data/websites_chunks_new.csv", index=False)
print(f"✅ Готово! {len(chunks_df)} чанков сохранено в 'data/websites_chunks.csv'")

✅ Готово! 11366 чанков сохранено в 'data/websites_chunks.csv'
